In [1]:
import os

# Fixes for loading datasets on Windows and keeps logs clean
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["DATASETS_VERBOSITY"] = "error"
os.environ["WANDB_DISABLED"] = "true"

import evaluate
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer
)

print("Libraries loaded safely. GPU Available:", torch.cuda.is_available())

Libraries loaded safely. GPU Available: True


In [2]:
print("Loading saved model and tokenizer from disk...")
model_path = "../newsmtsc_distilroberta_absa"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

print("Model loaded successfully!")

Loading saved model and tokenizer from disk...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Model loaded successfully!


In [3]:
print("Downloading test dataset...")
base_url = "https://raw.githubusercontent.com/fhamborg/NewsMTSC/6b838e00f54423c253806327a0ae24dbffa24c9e/NewsSentiment/experiments/default/datasets/newsmtsc-rw-hf/"
dataset = load_dataset("json", data_files={"test": base_url + "test.jsonl"})

print("Tokenizing test dataset...")
def tokenize_absa_function(examples):
    tokenized_inputs = tokenizer(
        examples["sentence"], 
        examples["mention"], 
        truncation=True, 
        max_length=128
    )
    tokenized_inputs["labels"] = [label + 1 for label in examples["polarity"]]
    return tokenized_inputs

tokenized_test = dataset["test"].map(tokenize_absa_function, batched=True)
print(f"Tokenization complete. Unseen test sentences loaded: {len(tokenized_test)}")

Generating test split: 0 examples [00:00, ? examples/s]

Tokenizing test dataset...


Map:   0%|          | 0/803 [00:00<?, ? examples/s]

Tokenization complete. Unseen test sentences loaded: 803


In [4]:
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return f1_metric.compute(predictions=predictions, references=labels, average="macro")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

print("\nRunning final evaluation on the test set...")
test_results = trainer.evaluate(eval_dataset=tokenized_test)

print("\n--- Final Test Set Results ---")
print(f"Test Loss: {test_results['eval_loss']:.4f}")
print(f"Test Macro F1-Score: {test_results['eval_f1'] * 100:.2f}%")


Running final evaluation on the test set...


Training Loss,Validation Loss,Step,F1
No log,0.619697,0,0.826939



--- Final Test Set Results ---
Test Loss: 0.6197
Test Macro F1-Score: 82.69%
